# 🛠️ JRA データ補完ツール
欠損している血統情報および過去走履歴を補完します。

In [2]:
# Google Driveをマウントする場合のみ実行してください
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ========================================
# 設定（ここを変更してください）
# ========================================
DATA_DIR = '/content/drive/MyDrive/dai-keiba/data/raw' # CSVがあるフォルダ

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import re
import time
import random
from datetime import datetime

class RaceScraper:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

    def _get_soup(self, url, max_retries=3):
        """Fetch URL with retry logic"""
        for attempt in range(max_retries):
            try:
                time.sleep(random.uniform(1.0, 2.0))
                response = requests.get(url, headers=self.headers, timeout=15)
                response.encoding = response.apparent_encoding
                if response.status_code == 200:
                    return BeautifulSoup(response.text, 'html.parser')
                elif attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"Status {response.status_code}, retrying in {wait_time}s...")
                    time.sleep(wait_time)
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"Error fetching {url}: {e}, retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"❌ Failed after {max_retries} attempts: {e}")
        return None

    def get_past_races(self, horse_id, n_samples=5):
        """
        Fetches past n_samples race results for a given horse_id from netkeiba db.
        Returns a DataFrame of past races.
        """
        url = f"https://db.netkeiba.com/horse/result/{horse_id}/"
        soup = self._get_soup(url)
        if not soup:
            return pd.DataFrame()

        # The results are usually in a table with class "db_h_race_results"
        table = soup.select_one("table.db_h_race_results")
        if not table:
            # Try to find any table with "着順"
            tables = soup.find_all("table")
            for t in tables:
                if "着順" in t.text:
                    table = t
                    break

        if not table:
            return pd.DataFrame()

        try:
            df = pd.read_html(io.StringIO(str(table)))[0]
            df = df.dropna(how='all')
            df.columns = df.columns.astype(str).str.replace(r'\s+', '', regex=True)

            if '日付' in df.columns:
                df['date_obj'] = pd.to_datetime(df['日付'], format='%Y/%m/%d', errors='coerce')
                df = df.dropna(subset=['date_obj'])
                df = df.sort_values('date_obj', ascending=False)

            if n_samples:
                df = df.head(n_samples)

            if '通過' in df.columns:
                df['run_style_val'] = df['通過'].apply(self.extract_run_style)
            else:
                df['run_style_val'] = 3

            column_map = {
                '日付': 'date',
                '開催': 'venue',
                '天気': 'weather',
                'レース名': 'race_name',
                '着順': 'rank',
                '枠番': 'waku',
                '馬番': 'umaban',
                '騎手': 'jockey',
                '斤量': 'weight_carried',
                '馬場': 'condition',
                'タイム': 'time',
                '着差': 'margin',
                '上り': 'last_3f',
                '通過': 'passing',
                '馬体重': 'horse_weight',
                'run_style_val': 'run_style',
                '単勝': 'odds',
                'オッズ': 'odds',
                '距離': 'raw_distance'
            }

            df.rename(columns=column_map, inplace=True)

            if 'raw_distance' in df.columns:
                def parse_dist(x):
                    if not isinstance(x, str): return None, None
                    surf = None
                    dist = None
                    if '芝' in x: surf = '芝'
                    elif 'ダ' in x: surf = 'ダ'
                    elif '障' in x: surf = '障'

                    match = re.search(r'(\d+)', x)
                    if match:
                        dist = int(match.group(1))
                    return surf, dist

                parsed = df['raw_distance'].apply(parse_dist)
                df['course_type'] = parsed.apply(lambda x: x[0])
                df['distance'] = parsed.apply(lambda x: x[1])
            else:
                df['course_type'] = None
                df['distance'] = None

            if 'rank' in df.columns:
                df['rank'] = pd.to_numeric(df['rank'], errors='coerce')

            if 'odds' in df.columns:
                 df['odds'] = pd.to_numeric(df['odds'], errors='coerce')

            for target_col in list(column_map.values()) + ['course_type', 'distance']:
                if target_col not in df.columns:
                    df[target_col] = None

            return df

        except Exception as e:
            print(f"Error parsing past races for {horse_id}: {e}")
            return pd.DataFrame()

    def extract_run_style(self, passing_str):
        """
        Converts passing order string (e.g., "1-1-1", "10-10-12") to run style (1,2,3,4).
        """
        if not isinstance(passing_str, str):
            return 3

        try:
            cleaned = re.sub(r'[^0-9-]', '', passing_str)
            parts = [int(p) for p in cleaned.split('-') if p]

            if not parts:
                return 3

            first_corner = parts[0]

            if first_corner == 1:
                return 1
            elif first_corner <= 4:
                return 2
            elif first_corner <= 9:
                return 3
            else:
                return 4

        except:
            return 3

    def get_horse_profile(self, horse_id):
        """
        Fetches horse profile to get pedigree (Father, Mother, Grandfather(BMS)).
        Returns a dictionary or None.
        """
        url = f"https://db.netkeiba.com/horse/ped/{horse_id}/"
        soup = self._get_soup(url)
        if not soup:
            return None

        data = {
            "father": "",
            "mother": "",
            "bms": ""
        }

        try:
            table = soup.select_one("table.blood_table")
            if table:
                rows = table.find_all("tr")

                if len(rows) >= 17:
                    r0 = rows[0].find_all("td")
                    if r0:
                        txt = r0[0].text.strip()
                        data["father"] = txt.split('\n')[0].strip()

                    r16 = rows[16].find_all("td")
                    if len(r16) >= 2:
                        m_txt = r16[0].text.strip()
                        data["mother"] = m_txt.split('\n')[0].strip()

                        bms_txt = r16[1].text.strip()
                        data["bms"] = bms_txt.split('\n')[0].strip()

        except Exception as e:
            print(f"Error parsing profile for {horse_id}: {e}")

        return data

    def get_race_metadata(self, race_id):
        """
        Fetches metadata for a specific race ID from Netkeiba.
        Returns dict with: race_name, date, venue, course_type, distance, weather, condition, turn
        """
        url = f"https://race.netkeiba.com/race/result.html?race_id={race_id}"
        soup = self._get_soup(url)
        if not soup:
            return None

        data = {
            "race_name": "",
            "date": "",
            "venue": "",
            "course_type": "",
            "distance": "",
            "weather": "",
            "condition": "",
            "turn": "",
            "race_id": race_id
        }

        try:
            title_elem = soup.select_one(".RaceName")
            if title_elem:
                data["race_name"] = title_elem.text.strip()

            rd1 = soup.select_one(".RaceData01")

            if rd1:
                txt = rd1.text.strip()

                if "天候:晴" in txt: data["weather"] = "晴"
                elif "天候:曇" in txt: data["weather"] = "曇"
                elif "天候:小雨" in txt: data["weather"] = "小雨"
                elif "天候:雨" in txt: data["weather"] = "雨"
                elif "天候:雪" in txt: data["weather"] = "雪"

                if "馬場:良" in txt: data["condition"] = "良"
                elif "馬場:稍" in txt: data["condition"] = "稍重"
                elif "馬場:重" in txt: data["condition"] = "重"
                elif "馬場:不良" in txt: data["condition"] = "不良"

                match = re.search(r'(芝|ダ|障)(\d+)m', txt)
                if match:
                    ctype_raw = match.group(1)
                    if ctype_raw == "芝": data["course_type"] = "芝"
                    elif ctype_raw == "ダ": data["course_type"] = "ダート"
                    elif ctype_raw == "障": data["course_type"] = "障害"

                    data["distance"] = match.group(2)

                if "右" in txt: data["turn"] = "右"
                elif "左" in txt: data["turn"] = "左"
                elif "直線" in txt: data["turn"] = "直"

            if not data["date"]:
                 meta_title = soup.title.text if soup.title else ""
                 match_date = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)', meta_title)
                 if match_date:
                     data["date"] = match_date.group(1)

        except Exception as e:
            print(f"Error parsing metadata for {race_id}: {e}")

        return data

In [5]:
import pandas as pd
import numpy as np
import os
import sys
import time
import random
from datetime import datetime
from tqdm.auto import tqdm

def fill_bloodline_data(df_path, mode="JRA"):
    """
    Backfills missing bloodline data (father, mother, bms).
    """
    print(f"\n🐴 Starting Bloodline Backfill for {mode} ({os.path.basename(df_path)})")

    if not os.path.exists(df_path):
        print(f"❌ File not found: {df_path}")
        return

    # Load Data
    try:
        if df_path.endswith('.parquet'):
            df = pd.read_parquet(df_path)
        else:
            df = pd.read_csv(df_path, low_memory=False)
            if 'horse_id' in df.columns:
                df['horse_id'] = df['horse_id'].astype(str).str.replace(r'\.0$', '', regex=True)
            if 'race_id' in df.columns:
                df['race_id'] = df['race_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return

    for col in ['father', 'mother', 'bms']:
        if col not in df.columns:
            df[col] = None

    mask_missing = (df['father'].isna()) | (df['father'] == '') | (df['father'] == 'nan')

    if 'horse_id' not in df.columns:
        print("❌ 'horse_id' column missing.")
        return

    target_ids = df.loc[mask_missing, 'horse_id'].dropna().unique()
    target_ids = [hid for hid in target_ids if str(hid).isdigit()]

    total_targets = len(target_ids)
    print(f"🎯 Found {total_targets} horses with missing bloodline data.")
    print(f"Using random delays (1.0-2.0s) to avoid rate limiting")

    if total_targets == 0:
        print("✅ No missing bloodline data found.")
        return

    scraper = RaceScraper()

    print(f"🚀 Fetching data for {total_targets} horses...")

    CHUNK_SIZE = 500
    failed_horses = []
    total_processed = 0

    results = {}
    for i in range(0, total_targets, CHUNK_SIZE):
        chunk = target_ids[i:i+CHUNK_SIZE]
        print(f"\n📦 Processing chunk {i//CHUNK_SIZE + 1}/{(total_targets-1)//CHUNK_SIZE + 1}")

        for hid in tqdm(chunk, desc="  Fetching bloodlines", leave=False):
            try:
                data = scraper.get_horse_profile(hid)
                if data:
                    results[hid] = data
                    total_processed += 1
                else:
                    failed_horses.append(hid)

                # Session keepalive
                if total_processed % 10 == 0 and total_processed > 0:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")

            except Exception as e:
                failed_horses.append(hid)

        if len(results) > 0:
            print("  Applying updates to DataFrame...")
            f_map = {h: d.get('father') for h, d in results.items() if d}
            m_map = {h: d.get('mother') for h, d in results.items() if d}
            b_map = {h: d.get('bms') for h, d in results.items() if d}

            mask_chunk = df['horse_id'].isin(results.keys())

            df.loc[mask_chunk, 'father'] = df.loc[mask_chunk, 'horse_id'].map(f_map).fillna(df.loc[mask_chunk, 'father'])
            df.loc[mask_chunk, 'mother'] = df.loc[mask_chunk, 'horse_id'].map(m_map).fillna(df.loc[mask_chunk, 'mother'])
            df.loc[mask_chunk, 'bms'] = df.loc[mask_chunk, 'horse_id'].map(b_map).fillna(df.loc[mask_chunk, 'bms'])

            results = {}

            print(f"  💾 Saving progress...")
            if df_path.endswith('.parquet'):
                df.to_parquet(df_path, index=False)
            else:
                df.to_csv(df_path, index=False)

    print(f"\n{'='*50}")
    print(f"✅ Bloodline backfill complete")
    print(f"成功: {total_processed}件")
    print(f"失敗: {len(failed_horses)}件")


def fill_history_data(df_path, mode="JRA"):
    """
    Backfills missing past race history (past_1_date, etc.).
    """
    print(f"\n📜 Starting History Backfill for {mode} ({os.path.basename(df_path)})")

    if not os.path.exists(df_path):
        print(f"❌ File not found: {df_path}")
        return

    try:
        if df_path.endswith('.parquet'):
            df = pd.read_parquet(df_path)
        else:
            df = pd.read_csv(df_path, low_memory=False)
            if 'horse_id' in df.columns:
                df['horse_id'] = df['horse_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return

    if 'レース名' in df.columns:
        mask_shinba = df['レース名'].astype(str).str.contains('新馬|メイクデビュー', na=False)
    else:
        mask_shinba = False

    mask_missing = df['past_1_date'].isna() & (~mask_shinba)

    target_rows = df[mask_missing]
    target_ids = target_rows['horse_id'].unique()
    target_ids = [hid for hid in target_ids if str(hid).isdigit()]

    total_targets = len(target_ids)
    print(f"🎯 Found {len(target_rows)} rows ({total_targets} unique horses) missing history.")
    print(f"Using random delays (1.0-2.0s) to avoid rate limiting")

    if total_targets == 0:
        print("✅ No missing history found.")
        return

    scraper = RaceScraper()
    history_cache = {}

    print(f"🚀 Fetching history for {total_targets} horses...")

    CHUNK_SIZE = 500
    total_processed = 0

    for i in range(0, total_targets, CHUNK_SIZE):
        chunk_ids = target_ids[i:i+CHUNK_SIZE]
        print(f"\n📦 Processing chunk {i//CHUNK_SIZE + 1}/{(total_targets-1)//CHUNK_SIZE + 1}")

        for hid in tqdm(chunk_ids, desc="  Fetching histories", leave=False):
             try:
                 hist_df = scraper.get_past_races(hid)
                 if hist_df is not None and not hist_df.empty:
                     if 'date' in hist_df.columns:
                         hist_df['date_dt'] = pd.to_datetime(hist_df['date'], format='%Y/%m/%d', errors='coerce')
                     history_cache[hid] = hist_df
                     total_processed += 1

                 # Session keepalive
                 if total_processed % 10 == 0 and total_processed > 0:
                     print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")
             except:
                 pass

        print("  Applying history to missing rows...")

        chunk_mask = df['horse_id'].isin(chunk_ids) & mask_missing
        affected_indices = df[chunk_mask].index

        for idx in tqdm(affected_indices, desc="  Updating rows", leave=False):
            row = df.loc[idx]
            hid = row['horse_id']
            race_date_str = str(row['日付'])

            if hid not in history_cache: continue

            hist_df = history_cache[hid]
            if hist_df is None or hist_df.empty: continue

            try:
                race_date_str = race_date_str.replace('年','/').replace('月','/').replace('日','')
                current_date = pd.to_datetime(race_date_str, errors='coerce')

                if pd.isna(current_date): continue

                valid_hist = hist_df[hist_df['date_dt'] < current_date].copy()

                if valid_hist.empty: continue

                valid_hist = valid_hist.sort_values('date_dt', ascending=False).head(5)

                cols_map = {
                    'date': 'date', 'rank': 'rank', 'time': 'time', 'run_style': 'run_style',
                    'race_name': 'race_name', 'last_3f': 'last_3f', 'horse_weight': 'horse_weight',
                    'jockey': 'jockey', 'condition': 'condition', 'weather': 'weather',
                    'distance': 'distance', 'course_type': 'course_type', 'odds': 'odds'
                }

                for n, (_, h_row) in enumerate(valid_hist.iterrows()):
                    if n >= 5: break
                    prefix = f"past_{n+1}_"

                    for key, val_key in cols_map.items():
                         df.at[idx, prefix + key] = h_row.get(val_key)

            except Exception as e:
                pass

        history_cache = {}

        print(f"  💾 Saving progress...")
        if df_path.endswith('.parquet'):
             df.to_parquet(df_path, index=False)
        else:
             df.to_csv(df_path, index=False)

    print(f"\n{'='*50}")
    print("✅ History backfill complete")


def fill_race_metadata(df_path, mode="JRA"):
    """
    Backfills missing race metadata (course_type, distance, weather, condition).
    """
    print(f"\n🏟️ Starting Race Metadata Backfill for {mode} ({os.path.basename(df_path)})")

    if not os.path.exists(df_path):
        print(f"❌ File not found: {df_path}")
        return

    try:
        if df_path.endswith('.parquet'):
            df = pd.read_parquet(df_path)
        else:
            df = pd.read_csv(df_path, low_memory=False)
            if 'race_id' in df.columns:
                df['race_id'] = df['race_id'].astype(str).str.replace(r'\.0$', '', regex=True)
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return

    target_cols = ['コースタイプ', '距離', '天候', '馬場状態']
    for c in target_cols:
        if c not in df.columns:
            df[c] = None

    missing_mask = (df['コースタイプ'].isna()) | (df['コースタイプ'] == '') | \
                   (df['距離'].isna()) | (df['距離'] == '') | \
                   (df['天候'].isna()) | (df['天候'] == '')

    target_race_ids = df.loc[missing_mask, 'race_id'].unique()
    target_race_ids = [rid for rid in target_race_ids if str(rid).isdigit()]

    total_targets = len(target_race_ids)
    print(f"🎯 Found {total_targets} races with missing metadata.")
    print(f"Using random delays (1.0-2.0s) to avoid rate limiting")

    if total_targets == 0:
        print("✅ No missing metadata found.")
        return

    scraper = RaceScraper()
    results = {}

    print(f"🚀 Fetching metadata for {total_targets} races...")

    CHUNK_SIZE = 200
    total_processed = 0

    for i in range(0, total_targets, CHUNK_SIZE):
        chunk = target_race_ids[i:i+CHUNK_SIZE]
        print(f"\n📦 Processing chunk {i//CHUNK_SIZE + 1}/{(total_targets-1)//CHUNK_SIZE + 1}")

        for rid in tqdm(chunk, desc="  Fetching metadata", leave=False):
            try:
                data = scraper.get_race_metadata(rid)
                if data and data.get('course_type'):
                    results[rid] = data
                    total_processed += 1

                # Session keepalive
                if total_processed % 10 == 0 and total_processed > 0:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")
            except:
                pass

        if len(results) > 0:
            print("  Applying metadata updates...")
            mask = df['race_id'].isin(results.keys())

            c_map = {rid: d['course_type'] for rid, d in results.items() if d.get('course_type')}
            d_map = {rid: d['distance'] for rid, d in results.items() if d.get('distance')}
            w_map = {rid: d['weather'] for rid, d in results.items() if d.get('weather')}
            cond_map = {rid: d['condition'] for rid, d in results.items() if d.get('condition')}

            df.loc[mask, 'コースタイプ'] = df.loc[mask, 'race_id'].map(c_map).fillna(df.loc[mask, 'コースタイプ'])
            df.loc[mask, '距離'] = df.loc[mask, 'race_id'].map(d_map).fillna(df.loc[mask, '距離'])
            df.loc[mask, '天候'] = df.loc[mask, 'race_id'].map(w_map).fillna(df.loc[mask, '天候'])
            df.loc[mask, '馬場状態'] = df.loc[mask, 'race_id'].map(cond_map).fillna(df.loc[mask, '馬場状態'])

            results = {}

            print(f"  💾 Saving progress...")
            if df_path.endswith('.parquet'):
                df.to_parquet(df_path, index=False)
            else:
                df.to_csv(df_path, index=False)

    print(f"\n{'='*50}")
    print("✅ Race metadata backfill complete")

In [ ]:
# 実行ブロック
csv_path = os.path.join(DATA_DIR, 'database.csv')
if os.path.exists(csv_path):
    print(f'処理対象: {csv_path}')
    fill_bloodline_data(csv_path, mode='JRA')
    fill_history_data(csv_path, mode='JRA')
    fill_race_metadata(csv_path, mode='JRA')
else:
    print(f'{csv_path} が見つかりません。')
    print(f'現在のディレクトリ: {os.getcwd()}')
    if os.path.exists(DATA_DIR):
        print(f'{DATA_DIR} の中身: {os.listdir(DATA_DIR)}')
    else:
        print(f'{DATA_DIR} ディレクトリ自体が存在しません。')

処理対象: /content/drive/MyDrive/dai-keiba/data/raw/database.csv

🐴 Starting Bloodline Backfill for JRA (database.csv)
🎯 Found 15983 horses with missing bloodline data.
Using random delays (1.0-2.0s) to avoid rate limiting
🚀 Fetching data for 15983 horses...

📦 Processing chunk 1/32


  Fetching bloodlines:   0%|          | 0/500 [00:00<?, ?it/s]

[20:05:18] ✅ 10件処理完了
[20:05:46] ✅ 20件処理完了
[20:06:13] ✅ 30件処理完了
[20:06:42] ✅ 40件処理完了
[20:07:12] ✅ 50件処理完了
[20:07:41] ✅ 60件処理完了
[20:08:07] ✅ 70件処理完了
[20:08:33] ✅ 80件処理完了
[20:09:02] ✅ 90件処理完了
[20:09:29] ✅ 100件処理完了
[20:09:56] ✅ 110件処理完了
[20:10:25] ✅ 120件処理完了
[20:10:50] ✅ 130件処理完了
[20:11:18] ✅ 140件処理完了
[20:11:44] ✅ 150件処理完了
[20:12:13] ✅ 160件処理完了
[20:12:41] ✅ 170件処理完了
[20:13:07] ✅ 180件処理完了
[20:13:33] ✅ 190件処理完了
[20:14:00] ✅ 200件処理完了
[20:14:28] ✅ 210件処理完了
[20:14:59] ✅ 220件処理完了
[20:15:27] ✅ 230件処理完了
[20:15:55] ✅ 240件処理完了
[20:16:23] ✅ 250件処理完了
[20:16:52] ✅ 260件処理完了
[20:17:22] ✅ 270件処理完了
[20:17:52] ✅ 280件処理完了
[20:18:21] ✅ 290件処理完了
[20:18:50] ✅ 300件処理完了
[20:19:17] ✅ 310件処理完了
[20:19:45] ✅ 320件処理完了
[20:20:13] ✅ 330件処理完了
[20:20:42] ✅ 340件処理完了
[20:21:10] ✅ 350件処理完了
[20:21:39] ✅ 360件処理完了
[20:22:06] ✅ 370件処理完了
[20:22:34] ✅ 380件処理完了
[20:23:00] ✅ 390件処理完了
[20:23:26] ✅ 400件処理完了
[20:23:53] ✅ 410件処理完了
[20:24:18] ✅ 420件処理完了
[20:24:47] ✅ 430件処理完了
[20:25:18] ✅ 440件処理完了
[20:25:45] ✅ 450件処理完了
[20:26:13] ✅ 460件処理

  Fetching bloodlines:   0%|          | 0/500 [00:00<?, ?it/s]

[20:28:46] ✅ 510件処理完了
[20:29:12] ✅ 520件処理完了
[20:29:38] ✅ 530件処理完了
[20:30:09] ✅ 540件処理完了
[20:30:37] ✅ 550件処理完了
[20:31:05] ✅ 560件処理完了
[20:31:35] ✅ 570件処理完了
[20:32:02] ✅ 580件処理完了
[20:32:31] ✅ 590件処理完了
[20:32:59] ✅ 600件処理完了
[20:33:26] ✅ 610件処理完了
[20:33:52] ✅ 620件処理完了
[20:34:22] ✅ 630件処理完了
[20:34:49] ✅ 640件処理完了
[20:35:19] ✅ 650件処理完了
[20:35:47] ✅ 660件処理完了
[20:36:15] ✅ 670件処理完了
[20:36:44] ✅ 680件処理完了
[20:37:12] ✅ 690件処理完了
[20:37:38] ✅ 700件処理完了
[20:38:07] ✅ 710件処理完了
[20:38:37] ✅ 720件処理完了
[20:39:05] ✅ 730件処理完了
[20:39:30] ✅ 740件処理完了
[20:40:00] ✅ 750件処理完了
[20:40:28] ✅ 760件処理完了
[20:40:55] ✅ 770件処理完了
[20:41:23] ✅ 780件処理完了
